In [1]:
import pandas as pd
import numpy as np
import pickle
import json
from sklearn.base import BaseEstimator, ClassifierMixin


class IsolationForestClassifier(BaseEstimator, ClassifierMixin):
    """
    Wrapper que transforma IsolationForest em um classificador binário
    compatível com RandomizedSearchCV e scoring='f1'.
    
    O IsolationForest retorna -1 (anomalia) e 1 (normal).
    Aqui mapeamos: anomalia (-1) → classe positiva (1), normal (1) → classe (0).
    """
    def __init__(self, n_estimators=100, max_samples='auto',
                 contamination=0.01, max_features=1.0,
                 bootstrap=False, random_state=None):
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.contamination = contamination
        self.max_features = max_features
        self.bootstrap = bootstrap
        self.random_state = random_state


    def fit(self, X, y=None):
        self.model_ = IsolationForest(
            n_estimators=self.n_estimators,
            max_samples=self.max_samples,
            contamination=self.contamination,
            max_features=self.max_features,
            bootstrap=self.bootstrap,
            random_state=self.random_state
        )
        self.model_.fit(X)
        
        # salva a régua do treino
        train_scores = self.model_.decision_function(X)
        self.score_min_ = train_scores.min()
        self.score_max_ = train_scores.max()
        return self
    
    def predict_proba(self, X):
        scores = self.model_.decision_function(X)
        
        # usa sempre a régua do treino
        prob_anomaly = 1 - (scores - self.score_min_) / (self.score_max_ - self.score_min_ + 1e-9)
        prob_anomaly = np.clip(prob_anomaly, 0, 1)
        return np.column_stack([1 - prob_anomaly, prob_anomaly])
        
    # def fit(self, X, y=None):
    #     self.model_ = IsolationForest(
    #         n_estimators=self.n_estimators,
    #         max_samples=self.max_samples,
    #         contamination=self.contamination,
    #         max_features=self.max_features,
    #         bootstrap=self.bootstrap,
    #         random_state=self.random_state
    #     )
    #     self.model_.fit(X)
    #     return self

    def predict(self, X):
        raw = self.model_.predict(X)          # -1 ou 1
        return np.where(raw == -1, 1, 0)      # anomalia → positivo

    # def predict_proba(self, X):
    #     scores = self.model_.decision_function(X)   # quanto mais baixo, mais anômalo
    #     # Normaliza para [0, 1] e inverte (score baixo = alta prob de ser anomalia)
    #     prob_anomaly = 1 - (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
    #     return np.column_stack([1 - prob_anomaly, prob_anomaly])

    def score(self, X, y):
        return f1_score(y, self.predict(X))


# import joblib          # alternativa ao pickle para carregar o modelo

# ==============================================================================
# 0. CONFIGURAÇÕES — ajuste os caminhos conforme seu ambiente
# ==============================================================================

PATH_SUBMISSAO   = "data/base_submissao.parquet"          # base de submissão
PATH_CLIENTES    = "data/base_cadastral.parquet"            # base de clientes (tipo_moradia)
PATH_MODELO      = "models/best_model.pkl"            # modelo serializado
PATH_WOE_MAPS    = "models/woe_maps.json"             # dicionário com os woe_maps do treino

PATH_OUTPUT      = "data/submissao.csv"    # arquivo de saída com scores

# Colunas numéricas que recebem WoE
WOE_FEATURES = ["valor_parcela", "valor_bem", "valor_credito", "renda_anual", "qtd_filhos"]

# Features finais usadas pelo modelo (exatamente como no notebook)
FEATURES = [
    'tipo_moradia_Municipal apartment',
    'tipo_moradia_Office apartment',
    'tipo_moradia_Rented apartment',
    'tipo_moradia_With parents',
    'tipo_contrato_Consumer loans',
    'tipo_contrato_Revolving loans',
    'valor_parcela_woe',
    'valor_bem_woe',
    'valor_credito_woe',
    'renda_anual_woe',
    'qtd_filhos_woe',
]



def apply_woe_from_map(series: pd.Series, woe_map: dict, bins: int = 10) -> pd.Series:
    """
    Aplica o mapeamento WoE a uma Series, usando os intervalos (bins) já
    calculados no treino.

    O woe_map pode conter:
      - chaves do tipo Interval (pd.qcut sobre numéricos)
      - chaves string (categóricos)

    Para numéricos, reconstruímos os intervalos a partir das chaves salvas
    em string e fazemos pd.cut com os breakpoints exatos.
    """
    if pd.api.types.is_numeric_dtype(series):
        # Reconstruir breakpoints a partir das chaves do woe_map
        # As chaves estão no formato "(a, b]" após serialização em JSON
        breakpoints = set()
        for key in woe_map:
            key_str = str(key).strip("([])")
            parts = key_str.split(",")
            if len(parts) == 2:
                try:
                    breakpoints.add(float(parts[0].strip()))
                    breakpoints.add(float(parts[1].strip()))
                except ValueError:
                    pass

        if breakpoints:
            bp = sorted(breakpoints)
            # Garante cobertura total: extende os limites
            bp[0]  = -np.inf
            bp[-1] =  np.inf
            binned = pd.cut(series, bins=bp, include_lowest=True)
            # Mapeia usando string para compatibilidade com JSON
            str_map = {str(k): v for k, v in woe_map.items()}
            return binned.astype(str).map(str_map)
        else:
            # Fallback: usa pd.qcut com o mesmo nº de bins do treino
            binned = pd.qcut(series, q=bins, duplicates="drop")
            str_map = {str(k): v for k, v in woe_map.items()}
            return binned.astype(str).map(str_map)
    else:
        return series.astype(str).map(woe_map)

print("=" * 60)
print("SCORING — BASE DE SUBMISSÃO")
print("=" * 60)

# --- 2.1 Modelo ---
print("\n[1/5] Carregando modelo...")
with open(PATH_MODELO, "rb") as f:
    model = pickle.load(f)
print(f"      Modelo carregado: {type(model).__name__}")

# --- 2.2 WoE maps ---
print("[2/5] Carregando woe_maps...")
with open(PATH_WOE_MAPS, "r") as f:
    woe_maps = json.load(f)
print(f"      Features com WoE: {list(woe_maps.keys())}")

# --- 2.3 Base de submissão ---
print("[3/5] Carregando base de submissão...")
df_sub = pd.read_parquet(PATH_SUBMISSAO)
print(f"      Shape: {df_sub.shape}")
print(f"      Colunas: {df_sub.columns.tolist()}")

# --- 2.4 Base de clientes (tipo_moradia) ---
print("[4/5] Carregando base de clientes...")
df_cli = pd.read_parquet(PATH_CLIENTES)
print(f"      Shape: {df_cli.shape}")


# ==============================================================================
# 3. PRÉ-PROCESSAMENTO
# ==============================================================================

print("\n[5/5] Pré-processando...")

# 3.1 Join com clientes para trazer tipo_moradia
df = df_sub.merge(
    df_cli[["id_cliente", "tipo_moradia", 'renda_anual', 'qtd_filhos']],
    on="id_cliente",
    how="left"
)
print(f"      Após join com clientes — shape: {df.shape}")

# 3.2 Preenchimento de nulos nas numéricas (igual ao notebook)
df["valor_bem"]     = df["valor_bem"].fillna(0)
df["valor_parcela"] = df["valor_parcela"].fillna(0)

# 3.3 One-Hot Encoding: tipo_moradia e tipo_contrato
#     drop_first=True foi usado no treino, então os dummies gerados são
#     calculados em relação à categoria de referência omitida.
COL_TO_DUMMIES = ["tipo_moradia", "tipo_contrato"]
df = pd.get_dummies(df, columns=COL_TO_DUMMIES, drop_first=True, dtype=int)

# Garante que todas as colunas dummy esperadas existam (pode faltar alguma
# categoria que não aparece na submissão)
for col in FEATURES:
    if col.startswith(("tipo_moradia_", "tipo_contrato_")) and col not in df.columns:
        df[col] = 0
        print(f"      ⚠️  Coluna dummy ausente adicionada com 0: {col}")

# 3.4 Aplicar WoE usando os mapas do treino
for feat in WOE_FEATURES:
    woe_col = f"{feat}_woe"
    if feat in woe_maps:
        df[woe_col] = apply_woe_from_map(df[feat], woe_maps[feat]).astype(float).fillna(0)
        print(f"      WoE aplicado: {feat} → {woe_col}")
    else:
        # Feature não tem woe_map salvo — preenche com 0 (neutro)
        df[woe_col] = 0.0
        print(f"      ⚠️  woe_map não encontrado para '{feat}' — preenchido com 0")

# 3.5 Verificação final das features
missing_feats = [f for f in FEATURES if f not in df.columns]
if missing_feats:
    raise ValueError(
        f"❌ Features ausentes no DataFrame antes do scoring:\n{missing_feats}\n"
        "Verifique o join e o encoding."
    )


# ==============================================================================
# 4. SCORING
# ==============================================================================

print("\n[6/5] Calculando scores...")
X_sub = df[FEATURES]

prob_default = model.predict_proba(X_sub)[:, 1]
pred_default = (prob_default >= 0.65).astype(int)

df["probabilidade_inadimplencia"] = prob_default # probabilidade de inadimplência
df["pred_default"] = pred_default          # classificação binária (0/1)

# Score invertido (0–1000): quanto maior, menos arriscado
df["score_risco"] = ((1 - prob_default) * 1000).round(0).astype(int)

print(f"\n      Distribuição do score de risco:")
print(df["score_risco"].describe().to_string())
print(f"\n      Predições de inadimplência:")
print(df["pred_default"].value_counts().rename({0: "Adimplente (0)", 1: "Inadimplente (1)"}).to_string())


# ==============================================================================
# 5. EXPORTAÇÃO
# ==============================================================================

# Colunas de saída

df_out = df[["id_cliente", "probabilidade_inadimplencia"]].copy()
df_out.to_csv(PATH_OUTPUT, index=False, sep=';')

print(f"\n✅ Arquivo de scores salvo em: {PATH_OUTPUT}")
print(f"   Shape: {df_out.shape}")
print("\nAmostra:")
print(df_out.head(10).to_string(index=False))
print("\n" + "=" * 60)
print("SCORING CONCLUÍDO")
print("=" * 60)

SCORING — BASE DE SUBMISSÃO

[1/5] Carregando modelo...
      Modelo carregado: DecisionTreeClassifier
[2/5] Carregando woe_maps...
      Features com WoE: ['valor_parcela', 'valor_bem', 'valor_credito', 'renda_anual', 'qtd_filhos']
[3/5] Carregando base de submissão...
      Shape: (40000, 8)
      Colunas: ['id_cliente', 'data_solicitacao', 'dia_semana_solicitacao', 'hora_solicitacao', 'tipo_contrato', 'valor_credito', 'valor_bem', 'valor_parcela']
[4/5] Carregando base de clientes...
      Shape: (40000, 16)

[5/5] Pré-processando...
      Após join com clientes — shape: (40000, 11)
      ⚠️  Coluna dummy ausente adicionada com 0: tipo_contrato_Consumer loans
      WoE aplicado: valor_parcela → valor_parcela_woe
      WoE aplicado: valor_bem → valor_bem_woe
      WoE aplicado: valor_credito → valor_credito_woe
      WoE aplicado: renda_anual → renda_anual_woe
      WoE aplicado: qtd_filhos → qtd_filhos_woe

[6/5] Calculando scores...

      Distribuição do score de risco:
count    4

In [2]:
df['pred_default'].value_counts(normalize=True)

pred_default
0    0.990975
1    0.009025
Name: proportion, dtype: float64

In [3]:
df['probabilidade_inadimplencia'].value_counts()

probabilidade_inadimplencia
0.618654    21849
0.209666     8035
0.498326     5842
0.000000     2367
0.140880     1222
0.602933      324
0.790127      226
0.919280      135
Name: count, dtype: int64